In [1]:
import pandas as pd
import numpy as np
import math
from itertools import combinations
from collections import defaultdict
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

#Read the data
df = pd.read_parquet(PROCESSED_DIR / "recipes.parquet")

In [7]:
#Get a quick overview of the dataset
print(df.info())

# Inspect a few ingredient lists
print(df["tags"].head())

<class 'pandas.DataFrame'>
Index: 228542 entries, 0 to 231636
Data columns (total 18 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   name            228542 non-null  str    
 1   id              228542 non-null  int64  
 2   minutes         228542 non-null  int64  
 3   contributor_id  228542 non-null  int64  
 4   submitted       228542 non-null  str    
 5   tags            228542 non-null  object 
 6   n_steps         228542 non-null  int64  
 7   steps           228542 non-null  object 
 8   description     228542 non-null  str    
 9   ingredients     228542 non-null  object 
 10  n_ingredients   228542 non-null  int64  
 11  calories        228542 non-null  float64
 12  total_fat       228542 non-null  float64
 13  sugar           228542 non-null  float64
 14  sodium          228542 non-null  float64
 15  protein         228542 non-null  float64
 16  sat_fat         228542 non-null  float64
 17  carbs           228542 non

In [2]:
# Collect all unique ingredients
all_ingredients = set()

for ingredients in df["ingredients"]:
    all_ingredients.update(ingredients)

# Assign a stable integer ID to each ingredient
# First sort ingredients alphabetically to ensure consistent ordering
sorted_ingredients = sorted(all_ingredients)

ingredient_to_id = {
    ingredient: idx
    for idx, ingredient in enumerate(sorted_ingredients)
}

id_to_ingredient = {
    idx: ingredient
    for ingredient, idx in ingredient_to_id.items()
}

In [3]:
# Collect all unique tags
all_tags = set()

for tags in df["tags"]:
    all_tags.update(tags)

# Assign a stable integer ID to each tag
# First sort tags alphabetically to ensure consistent ordering
sorted_tags = sorted(all_tags)

tag_to_id = {
    tag: idx
    for idx, tag in enumerate(sorted_tags)
}

id_to_tag = {
    idx: tag
    for tag, idx in tag_to_id.items()
}

In [14]:
#Small check with one ingredient to see that id and ingredient match.
print("Unique ingredients:", len(ingredient_to_id))
print(ingredient_to_id["salt"])  # Example: Get the ID for "salt" get 11644
print(id_to_ingredient[11644])  # Example: Get the ingredient for ID 11644 get "salt"

Unique ingredients: 14801
11644
salt


In [15]:
baskets = []

for ingredients in df["ingredients"]:
    item_ids = {
        ingredient_to_id[item]
        for item in ingredients
    }
    baskets.append(sorted(item_ids))


In [9]:
baskets = []

for tags in df["tags"]:
    item_ids = {
        tag_to_id[tag]
        for tag in tags
    }
    baskets.append(sorted(item_ids))

In [16]:
#Small check if the mapping of the baskets is correct.
for id in baskets[0]:
    print(id_to_ingredient[id])  # Print the ingredient corresponding to each ID in the first basket
print(df["ingredients"].iloc[0])  # Print the original ingredient list for the first basket
#We see they coincide, so the mapping is correct.

butter
honey
mexican seasoning
mixed spice
olive oil
salt
winter squash
['winter squash' 'mexican seasoning' 'mixed spice' 'honey' 'butter'
 'olive oil' 'salt']


In [10]:
#We now start by setting up an array to count the number of occurrences of each ingredient across all baskets. 
#num_items = len(ingredient_to_id)
num_items = len(tag_to_id)

singleton_counts = np.zeros(num_items, dtype=np.int64)

for basket in baskets:
    for item_id in basket:
        singleton_counts[item_id] += 1

In [17]:
#Some statistics
print("Number of recipes:", len(baskets))
print("Number of unique ingredients:", num_items)

print(
    "Average ingredients per recipe:",
    np.mean([len(basket) for basket in baskets])
)

print(
    "Total ingredient occurrences:",
    singleton_counts.sum()
)

print(
    "Average occurrences per ingredient:",
    np.mean(singleton_counts)
)

Number of recipes: 228542
Number of unique ingredients: 552
Average ingredients per recipe: 9.051885430249145
Total ingredient occurrences: 4087383
Average occurrences per ingredient: 7404.679347826087


In [11]:
#Some statistics
print("Number of recipes:", len(baskets))
print("Number of unique tags:", num_items)

print(
    "Average tags per recipe:",
    np.mean([len(basket) for basket in baskets])
)

print(
    "Total tag occurrences:",
    singleton_counts.sum()
)

print(
    "Average occurrences per tag:",
    np.mean(singleton_counts)
)

Number of recipes: 228542
Number of unique tags: 552
Average tags per recipe: 17.884603267670713
Total tag occurrences: 4087383
Average occurrences per tag: 7404.679347826087


In [9]:
top_items = np.argsort(singleton_counts)[::-1][:20]

for item_id in top_items:
    ingredient = id_to_ingredient[item_id]
    count = singleton_counts[item_id]

    print(f"{ingredient}: {count}")

salt: 84627
butter: 54472
sugar: 43700
onion: 38644
water: 34245
eggs: 33405
olive oil: 32490
flour: 25987
milk: 25541
garlic cloves: 25475
pepper: 22066
brown sugar: 18372
garlic: 17673
all-purpose flour: 17478
baking powder: 17346
egg: 17148
salt and pepper: 15305
parmesan cheese: 14717
lemon juice: 13980
baking soda: 13968


In [20]:
#We recognize that some most common ingrediets are redundant, e.g. "salt" and "salt and pepper".
#Hence let us do a short check to see how many such edge cases we have.
# Search for ingredient descriptions containing "and"
and_ingredients = [
    ingredient
    for ingredient in all_ingredients
    if " and " in ingredient.lower()
]

print("Number of descriptions containing 'and':", len(and_ingredients))
for ingredient in and_ingredients:
    print(ingredient)

Number of descriptions containing 'and': 357
frozen mixed potatoes carrots celery and onions
lemon, juice and zest of
mushroom and garlic couscous
crunchy sweetened corn and oat cereal
frozen peas and carrot
stewed tomatoes with peppers and celery
kraft tuscan house italian dressing and marinade
ragu parmesan and romano spaghetti sauce
sour cream and onion potato chips
lipton beef-flavor noodles and sauce
fruit and bran cereal
cheddar and american blend cheese
good seasons garlic and herb seasoning mix
maple and brown sugar flavored hot wheat cereal mix
diced tomatoes with lime and cilantro
diced tomatoes with basil oregano and garlic
vegan sweet and sour sauce
tomato sauce with basil garlic and oregano
cabbage and carrot coleslaw mix
split pea and ham soup
shredded steak and cheese taquitos
mrs. dash garlic and herb seasoning
onion and garlic croutons
lime, juice and zest of
chocolate flavor instant pudding and pie filling mix
greek vinaigrette dressing with feta and oregano
cream che

In [21]:
for keyword in ["salt", "pepper", "garlic", "egg", "flour", "rice", "sugar", "butter", "milk", "onion"]:
    print(f"\n--- {keyword.upper()} ---")

    matches = [
        ingredient
        for ingredient in all_ingredients
        if keyword in ingredient.lower()
    ]

    for ingredient in sorted(matches):
        print(ingredient)



--- SALT ---
aunt jane's krazy mixed up salt
black salt
butter-flavored salt
canned salt-reduced spaghetti
canning salt
capers packed in salt
cavenders salt-free all purpose greek seasoning
celery salt
chex mix sweet and salty trail mix
chicken salt
chinese salt
coarse salt
coarse sea salt
crazy salt
curing salt
dried salt cod fish
dry roasted salted peanut
dry roasted salted peanuts
dry-roasted salted macadamia nuts
dry-roasted salted mixed nuts
dry-roasted unsalted peanuts
epsom salts
fat salt pork
fat-free saltine crackers
fine salt
fine sea salt
flaked sea salt
fruit salt
garlic salt
garlic salt with parsley
garlic sea salt
gray salt
hawaiian sea salt
herb-seasoned salt
herbal salt substitute
herbamare herb seasoning salt
hickory smoke salt
hot salt
hunt's no-salt-added tomato paste
hunt's tomato sauce-no salt added
iodized salt
j&d peppered bacon salt
jalapeno seasoning salt
kosher salt
kosher salt & freshly ground black pepper
lactose-free unsalted margarine
large capers packed 

Although we have to deal with this issues we first implement the framework for the PCY-algorithm and come back later - if one get unreasonable results - to this issue.

In [27]:
from nltk.stem import WordNetLemmatizer
import nltk
import re
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

lemmatizer = WordNetLemmatizer()

def normalize_ingredient(ingredient):
    ingredient = ingredient.lower().strip()
    ingredient = re.sub(r"\s+", " ", ingredient)

    words = ingredient.split()

    normalized_words = [
        lemmatizer.lemmatize(word, pos="n")
        for word in words
    ]

    return " ".join(normalized_words)


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\andre\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [28]:
print(normalize_ingredient(" yellow sweet onions"))  # Output: "tomato"

yellow sweet onion


In [11]:
#We use as starting parameters for the PCY algorithm the following thresholds:
min_support = 1000  # Minimum number of occurrences for an ingredient to be considered frequent
num_buckets = 100000  # Number of buckets for hashing pairs of ingredients (can be increased)
#Additionally we need a hash-function to map pairs of ingredients to buckets.
def hash_pair(i, j):
    return hash((min(i, j), max(i, j))) % num_buckets

In [12]:
# We redo the counting of singletons to have it all in one cell
# Count occurrences of individual ingredients
singleton_counts = np.zeros(num_items, dtype=np.int64)

# Count occurrences of hashed ingredient pairs
bucket_counts = np.zeros(num_buckets, dtype=np.int64)

for basket in baskets:

    # Count individual ingredients
    for item in basket:
        singleton_counts[item] += 1

    # Hash every pair into a bucket
    for item1, item2 in combinations(basket, 2):
        bucket = hash_pair(item1, item2)
        bucket_counts[bucket] += 1

In [13]:
print("Total singleton occurrences:", singleton_counts.sum())
print("Total pair occurrences:", bucket_counts.sum())

print("Number of non-empty buckets:", np.count_nonzero(bucket_counts))
print("Maximum bucket count:", bucket_counts.max())
print("Average bucket count:", bucket_counts.mean())

Total singleton occurrences: 2068736
Total pair occurrences: 9917661
Number of non-empty buckets: 100000
Maximum bucket count: 25390
Average bucket count: 99.17661


In [ ]:
# Create the PCY bitmap
frequent_buckets = bucket_counts >= min_support

print("Number of frequent buckets:", np.count_nonzero(frequent_buckets))
print("Number of infrequent buckets:", np.count_nonzero(~frequent_buckets))

# Identify frequent individual ingredients
frequent_items = singleton_counts >= min_support

print("Number of frequent individual ingredients:",
      np.count_nonzero(frequent_items))

Number of frequent buckets: 824
Number of infrequent buckets: 99176
Number of frequent individual ingredients: 315


In [17]:
pair_counts = defaultdict(int)

for basket in baskets:

    # Keep only frequent individual ingredients
    frequent_basket = [
        item for item in basket
        if frequent_items[item]
    ]

    # Generate pairs from the remaining ingredients
    for i, j in combinations(frequent_basket, 2):

        bucket = hash_pair(i, j)

        # Only count pairs hashing to frequent buckets
        if frequent_buckets[bucket]:
            pair_counts[(i, j)] += 1

In [18]:
frequent_pairs = {
    pair: count
    for pair, count in pair_counts.items()
    if count >= min_support
}

print("Number of candidate pairs counted:", len(pair_counts))
print("Number of frequent pairs:", len(frequent_pairs))

Number of candidate pairs counted: 1259
Number of frequent pairs: 735


In [22]:
top_pairs = sorted(
    frequent_pairs.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

for (i, j), count in top_pairs:
    print(
        f"{id_to_ingredient[i]} + {id_to_ingredient[j]}: {count}"
    )

butter + salt: 25335
salt + sugar: 22362
eggs + salt: 17928
pepper + salt: 17392
onion + salt: 16437
flour + salt: 16235
salt + water: 15170
butter + sugar: 14885
butter + eggs: 14071
butter + flour: 13720
olive oil + salt: 13194
baking powder + salt: 13115
eggs + sugar: 12808
milk + salt: 12751
all-purpose flour + salt: 12090
butter + milk: 11598
flour + sugar: 11006
baking soda + salt: 10672
garlic cloves + salt: 10492
egg + salt: 9709


In [23]:
# Set of all frequent pairs
frequent_pair_set = set(frequent_pairs.keys())

# Group pairs by their first ingredient
pairs_by_first = defaultdict(list)

for i, j in frequent_pair_set:
    pairs_by_first[i].append(j)

# Generate candidate triples
candidate_triples = set()

for i, items in pairs_by_first.items():

    items = sorted(items)

    for j, k in combinations(items, 2):

        # Check whether the third pair is also frequent
        if (j, k) in frequent_pair_set:
            candidate_triples.add((i, j, k))

print("Number of candidate triples:", len(candidate_triples))

Number of candidate triples: 2404


In [24]:
triple_counts = defaultdict(int)

for basket in baskets:

    # Keep only frequent individual ingredients
    frequent_basket = [
        item for item in basket
        if frequent_items[item]
    ]

    # Generate all triples in this recipe
    for triple in combinations(frequent_basket, 3):

        # Count only candidate triples
        if triple in candidate_triples:
            triple_counts[triple] += 1

In [25]:
frequent_triples = {
    triple: count
    for triple, count in triple_counts.items()
    if count >= min_support
}

print("Number of candidate triples:", len(candidate_triples))
print("Number of frequent triples:", len(frequent_triples))

Number of candidate triples: 2404
Number of frequent triples: 518


In [26]:
top_triples = sorted(
    frequent_triples.items(),
    key=lambda x: x[1],
    reverse=True
)[:20]

for (i, j, k), count in top_triples:
    ingredients = [
        id_to_ingredient[i],
        id_to_ingredient[j],
        id_to_ingredient[k]
    ]

    print(f"{ingredients}: {count}")

['butter', 'salt', 'sugar']: 8718
['butter', 'flour', 'salt']: 8364
['eggs', 'salt', 'sugar']: 8092
['butter', 'eggs', 'salt']: 8042
['flour', 'salt', 'sugar']: 7765
['baking powder', 'salt', 'sugar']: 7233
['butter', 'milk', 'salt']: 6713
['butter', 'eggs', 'sugar']: 6710
['baking powder', 'eggs', 'salt']: 6489
['onion', 'pepper', 'salt']: 6200
['butter', 'flour', 'sugar']: 6180
['eggs', 'flour', 'salt']: 6176
['baking powder', 'butter', 'salt']: 5843
['baking soda', 'eggs', 'salt']: 5730
['baking soda', 'salt', 'sugar']: 5681
['baking powder', 'flour', 'salt']: 5633
['eggs', 'flour', 'sugar']: 5379
['all-purpose flour', 'butter', 'salt']: 5377
['baking powder', 'baking soda', 'salt']: 5293
['milk', 'salt', 'sugar']: 5188
